# Evidence-Grounded Policy Research Assistant Walkthrough
### End-to-End Demonstration: Mixed-Format Ingestion, Hybrid RAG, Deterministic Calculations, Guardrails, and Evaluation

This interactive walkthrough demonstrates:
1. Ingestion of structured statistics (CSV) and unstructured policy whitepapers (Markdown).
2. Hybrid retrieval (BM25 + Dense Semantic RRF) and heuristic re-ranking.
3. Deterministic calculation tool calling (CAGR, percentages) without LLM hallucination.
4. Principled abstention on out-of-scope queries and prompt injection resistance.
5. Conflicting evidence detection and empirical reconciliation.
6. Model Context Protocol (MCP) tool discovery and invocation.

In [ ]:
import sys
from pathlib import Path
# Ensure project root is in python path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from src.agent.workflow import PolicyAgentWorkflow
from src.mcp.client import PolicyMCPClient
from src.evaluation.runner import BenchmarkRunner

workflow = PolicyAgentWorkflow()
print(f"Corpus initialized with {len(workflow.corpus_chunks)} chunks across structured and unstructured sources.")

## 1. Factual Query Answering with Traceable Citations

In [ ]:
q1 = "What was the Value Added (VA) generated by Singapore's Infocomm Media sector in 2023?"
ans1, state1 = workflow.run(q1)
print("Direct Answer:", ans1.direct_answer)
print("\nCitations:")
for c in ans1.citations:
    print(f"- [{c.source_id}] {c.section}: \"{c.quote}\"")

## 2. Deterministic Calculation (Zero Arithmetic Hallucination)

In [ ]:
q2 = "Calculate the Compound Annual Growth Rate (CAGR) of Singapore's Infocomm Media Value Added from 2018 (SGD 19.2B) to 2023 (SGD 38.6B)."
ans2, state2 = workflow.run(q2)
print("Direct Answer:", ans2.direct_answer)
print("Verified Calculations:", [c.model_dump() for c in ans2.calculations])

## 3. Principled Abstention on Out-of-Scope Queries

In [ ]:
q3 = "What are Singapore's agricultural rice production quotas for 2024?"
ans3, state3 = workflow.run(q3)
print("Response:", ans3.direct_answer)
print("Abstention Status:", ans3.abstention.model_dump() if ans3.abstention else None)
print("Uncertainty Level:", ans3.uncertainty_level)

## 4. Conflicting Evidence Reconciliation

In [ ]:
q4 = "What is the projected net impact of generative AI on Singapore tech jobs by 2027?"
ans4, state4 = workflow.run(q4)
print("Response:", ans4.direct_answer)
if ans4.conflict_analysis:
    print("\n--- Conflict Reconciliation Analysis ---")
    print(f"Source A: {ans4.conflict_analysis.source_a} -> {ans4.conflict_analysis.claim_a}")
    print(f"Source B: {ans4.conflict_analysis.source_b} -> {ans4.conflict_analysis.claim_b}")
    print(f"Guidance: {ans4.conflict_analysis.reconciliation_guidance}")

## 5. Model Context Protocol (MCP) Capability Discovery

In [ ]:
mcp_client = PolicyMCPClient()
caps = mcp_client.discover_capabilities()
print("Discovered MCP Tools:", caps["tools"])
print("Discovered MCP Resources:", caps["resources"])
res = mcp_client.call_tool("calculate_growth_rate", {"start_val": 19.2, "end_val": 38.6, "num_years": 5.0})
print("\nMCP Tool Call Result:", res["data"])
print("Security Envelope:", res["security_envelope"][:200], "...")